# 05 - Model Comparison (HOG + KNN / Decision Tree / SVM)

Project: **RoadEye - Traffic Sign Recognition Using HOG Features and SVM** (Machine Learning Innovative Assignment).

> Run cells top to bottom. All numbers are computed when the cell runs; nothing is typed in by hand.

In [ ]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
from src import config, pipeline as P
np.random.seed(config.SEED)                      # reproducibility
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("RUN MODE:", config.RUN_MODE, "| seed:", config.SEED)

In [ ]:
def show(name, width=900):
    """Display a saved figure from results/figures/."""
    display(Image(filename=str(config.FIGURES_DIR / name), width=width))

def metrics(name):
    return json.loads((config.METRICS_DIR / name).read_text())

## Fair comparison protocol
Same train/validation split, same HOG features, same StandardScaler, same metrics. Winner is chosen **automatically** by macro-F1 on the validation set (`PRIMARY_METRIC`). The test set is not used here.

In [ ]:
data = P.stage_prepare(save_figures=False)
if not (config.METRICS_DIR / 'baseline_svm.json').exists():
    P.stage_baseline(data)
df = P.stage_compare(data)
cols = ['model','feature_representation','accuracy','precision_macro','recall_macro','f1_macro','training_time_seconds','inference_time_seconds','train_accuracy']
display(df[cols].round(4))

In [ ]:
rq = json.loads((config.COMPARISON_DIR / 'research_questions.json').read_text())
print('RQ2 winner among untuned HOG classifiers:', rq['RQ2_best_untuned_hog_classifier'])
print('RQ1 raw vs HOG SVM:', rq['RQ1_raw_vs_hog_svm'])
show('model_performance_comparison.png', 800)

## Does class weighting help? (validation only)

In [ ]:
print(metrics('balancing.json')['decision'])

**Interpretation guide:** KNN suffers from the curse of dimensionality (1764-d) and gets no class weights; a single Decision Tree is high-variance and axis-aligned; the RBF SVM can model smooth non-linear boundaries in the high-dimensional HOG space. Whether this holds is decided by the numbers above.